In [25]:
import cv2
import mediapipe as mp
import numpy as np
import math

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands


# For webcam input:
cap = cv2.VideoCapture('hand5.mp4')
현재색 = (0, 0, 255)   # 시작 색: 빨강
이전점 = None
캔버스 = None
fire_cap = cv2.VideoCapture('fire1.mp4')
점들 = []
with mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as hands:
  while cap.isOpened():
    success, image = cap.read()
    if not success:
      print("더 남은 프레임 없다잉.")
      # If loading a video, use 'break' instead of 'continue'.
      break
    for _ in range(15):
        success2, fire = fire_cap.read()
        if not success2:
            fire_cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            success2, fire = fire_cap.read()
        fire = cv2.resize(fire, (100, 100))

    # To improve performance, optionally mark the image as not writeable to
    # pass by reference.
    image.flags.writeable = False
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = hands.process(image)

    # Draw the hand annotations on the image.
    image.flags.writeable = True
    높이, 너비, 채널 = image.shape
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    if 캔버스 is None:
        캔버스 = np.ones((높이, 너비, 3), dtype=np.uint8) * 255
 # 1. 위쪽 메뉴바
        # -------------------------
    cv2.rectangle(image, (0, 0), (너비, 100), (200, 200, 200), -1)

    # 색 버튼 3개 + clear
    cv2.rectangle(image, (10, 10), (90, 90), (0, 0, 255), -1)     # 빨강
    cv2.rectangle(image, (110, 10), (190, 90), (0, 255, 0), -1)   # 초록
    cv2.rectangle(image, (210, 10), (290, 90), (255, 0, 0), -1)   # 파랑
    cv2.rectangle(image, (310, 10), (430, 90), (255, 255, 255), -1)  # clear

    cv2.putText(image, "CLEAR", (325, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)

    if results.multi_hand_landmarks:
      for hand_landmarks in results.multi_hand_landmarks:
        mp_drawing.draw_landmarks( image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
        #     mp_drawing_styles.get_default_hand_landmarks_style(),
        #     mp_drawing_styles.get_default_hand_connections_style())

        엄지포인트 = hand_landmarks.landmark[4]
        검지포인트 = hand_landmarks.landmark[8]
        x1 = int(엄지포인트.x * 너비)
        y1 = int(엄지포인트.y * 높이)
        x2 = int(검지포인트.x * 너비)
        y2 = int(검지포인트.y * 높이)

        거리 = math.hypot(x1-x2, y1-y2) 
        cv2.putText(image, f"distance: {int(거리)}", (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.line(image, (x1, y1), (x2, y2), (255, 0, 255), 3)  # 두 점 사이 선
          
        if 거리 < 100:
            cv2.putText(image, "CLICK", (50, 100),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
            
        cv2.circle(image, (x2, y2), 8, (0, 0, 255), -1)
        if y2 < 100:
            이전점 = None

            if 10 <= x2 <= 90:
                현재색 = (0, 0, 255)   # 빨강

            elif 110 <= x2 <= 190:
                현재색 = (0, 255, 0)   # 초록

            elif 210 <= x2 <= 290:
                현재색 = (255, 0, 0)   # 파랑

            elif 310 <= x2 <= 430:
                캔버스[:] = 255        # 전체 삭제

        # -------------------------
        # 3. 메뉴 아래에서는
        # 엄지+검지 가까울 때만 그림 그리기
        # -------------------------
        else:
            if 거리 < 150:
                cv2.putText(image, "DRAW", (50, 180),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

                if 이전점 is None:
                    이전점 = (x2, y2)
                else:
                    cv2.line(캔버스, 이전점, (x2, y2), 현재색, 5)
                    이전점 = (x2, y2)
            else:
                이전점 = None

    else:
        이전점 = None

        # -------------------------
        # 4. 캔버스랑 원본 합치기
        # -------------------------
    결과 = cv2.addWeighted(image, 0.7, 캔버스, 0.3, 0)
  
    #     점들.append((x,y))
    #     cv2.circle(image, (x,y), 5, (0,0,255),-1)    

    # for i in range(1, len(점들)):
    #     cv2.line(image, 점들[i-1], 점들[i], (0,0,255), 3)
          
          
    # Flip the image horizontally for a selfie-view display.
    cv2.imshow('MediaPipe Hands', 결과)
    if cv2.waitKey(30) & 0xFF == ord('q'):
      break
cap.release()
cv2.destroyAllWindows()

In [43]:
import cv2
import mediapipe as mp
import numpy as np
import math

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

cap = cv2.VideoCapture('hand5.mp4')

현재색 = (0, 0, 255)
이전점 = None
캔버스 = None

with mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print("더 남은 프레임 없다잉.")
            break

        image.flags.writeable = False
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        image.flags.writeable = True
        높이, 너비, 채널 = image.shape

        if 캔버스 is None:
            캔버스 = np.zeros((높이, 너비, 3), dtype=np.uint8)

        # 메뉴바
        cv2.rectangle(image, (0, 0), (너비, 100), (200, 200, 200), -1)
        cv2.rectangle(image, (10, 10), (90, 90), (0, 0, 255), -1)
        cv2.rectangle(image, (110, 10), (190, 90), (0, 255, 0), -1)
        cv2.rectangle(image, (210, 10), (290, 90), (255, 0, 0), -1)
        cv2.rectangle(image, (310, 10), (430, 90), (255, 255, 255), -1)
        cv2.putText(image, "CLEAR", (325, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)

                엄지포인트 = hand_landmarks.landmark[4]
                검지포인트 = hand_landmarks.landmark[8]

                x1 = int(엄지포인트.x * 너비)
                y1 = int(엄지포인트.y * 높이)
                x2 = int(검지포인트.x * 너비)
                y2 = int(검지포인트.y * 높이)

                거리 = math.hypot(x1 - x2, y1 - y2)

                cv2.circle(image, (x2, y2), 8, (0, 0, 255), -1)
                cv2.putText(image, f"distance: {int(거리)}", (30, 140),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

                if y2 < 100:
                    이전점 = None

                    if 10 <= x2 <= 90:
                        현재색 = (0, 0, 255)
                    elif 110 <= x2 <= 190:
                        현재색 = (0, 255, 0)
                    elif 210 <= x2 <= 290:
                        현재색 = (255, 0, 0)
                    elif 310 <= x2 <= 430:
                        캔버스[:] = 0

                else:
                    if 거리 < 220:
                        cv2.putText(image, "DRAW", (50, 200),
                                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

                        if 이전점 is None:
                            이전점 = (x2, y2)
                        else:
                            cv2.line(캔버스, 이전점, (x2, y2), 현재색, 8)
                            이전점 = (x2, y2)
                    else:
                        이전점 = None
        else:
            이전점 = None

        결과 = cv2.addWeighted(image, 1.0, 캔버스, 1.0, 0)

        cv2.imshow('MediaPipe Hands', 결과)
        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [53]:
import cv2
import mediapipe as mp
import numpy as np

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

cap = cv2.VideoCapture('hand5.mp4')

현재색 = (0, 0, 255)   # 시작: 빨강
현재도구 = "pen"        # pen 또는 eraser
이전점 = None
캔버스 = None

with mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print("더 남은 프레임 없다잉.")
            break

        # 손 인식용 RGB 변환
        image.flags.writeable = False
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        image.flags.writeable = True
        높이, 너비, 채널 = image.shape

        # 캔버스 처음 생성
        if 캔버스 is None:
            캔버스 = np.ones((높이, 너비, 3), dtype=np.uint8) * 255

        # -------------------------
        # 1. 메뉴바
        # -------------------------
        cv2.rectangle(image, (0, 0), (너비, 100), (220, 220, 220), -1)

        # 색상 버튼
        cv2.rectangle(image, (10, 10), (90, 90), (0, 0, 255), -1)     # 빨강
        cv2.rectangle(image, (110, 10), (190, 90), (0, 255, 0), -1)   # 초록
        cv2.rectangle(image, (210, 10), (290, 90), (255, 0, 0), -1)   # 파랑

        # 도구 버튼
        cv2.rectangle(image, (310, 10), (430, 90), (180, 180, 180), -1)  # 지우개
        cv2.putText(image, "ERASE", (325, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

        cv2.rectangle(image, (450, 10), (570, 90), (255, 255, 255), -1)  # 전체삭제
        cv2.putText(image, "CLEAR", (465, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

        # 현재 선택 상태 표시
        if 현재도구 == "pen":
            cv2.rectangle(image, (600, 10), (680, 90), 현재색, -1)
            cv2.putText(image, "PEN", (610, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        else:
            cv2.rectangle(image, (600, 10), (680, 90), (200, 200, 200), -1)
            cv2.putText(image, "ERASE", (605, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)

        # -------------------------
        # 2. 손 인식
        # -------------------------
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)

                # 검지 끝 좌표 (8번)
                검지포인트 = hand_landmarks.landmark[8]
                x = int(검지포인트.x * 너비)
                y = int(검지포인트.y * 높이)

                # 손가락 끝 표시
                cv2.circle(image, (x, y), 8, (0, 0, 255), -1)

                # -------------------------
                # 3. 메뉴 선택
                # -------------------------
                if y < 100:
                    이전점 = None

                    if 10 <= x <= 90:
                        현재색 = (0, 0, 255)
                        현재도구 = "pen"

                    elif 110 <= x <= 190:
                        현재색 = (0, 255, 0)
                        현재도구 = "pen"

                    elif 210 <= x <= 290:
                        현재색 = (255, 0, 0)
                        현재도구 = "pen"

                    elif 310 <= x <= 430:
                        현재도구 = "eraser"

                    elif 450 <= x <= 570:
                        캔버스[:] = 255

                # -------------------------
                # 4. 그리기 / 지우기
                # -------------------------
                else:
                    if 이전점 is None:
                        이전점 = (x, y)
                    else:
                        if 현재도구 == "pen":
                            cv2.line(캔버스, 이전점, (x, y), 현재색, 5)
                            cv2.putText(image, "STATE: PEN", (20, 140), cv2.FONT_HERSHEY_SIMPLEX, 1, 현재색, 2)
                        elif 현재도구 == "eraser":
                            cv2.line(캔버스, 이전점, (x, y), (255, 255, 255), 25)
                            cv2.putText(image, "STATE: ERASER", (20, 140), cv2.FONT_HERSHEY_SIMPLEX, 1, (100, 100, 100), 2)

                        이전점 = (x, y)

        else:
            이전점 = None

        # -------------------------
        # 5. 원본 화면 + 캔버스 합치기
        # -------------------------
        결과 = image.copy()

        # 흰색이 아닌 부분만 덮어쓰기
        mask = np.any(캔버스 != 255, axis=2)
        결과[mask] = 캔버스[mask]

        cv2.imshow('BLACK BOARD', 결과)

        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

더 남은 프레임 없다잉.


In [56]:
import cv2
import mediapipe as mp
import numpy as np

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

cap = cv2.VideoCapture('hand5.mp4')

현재색 = (0, 0, 255)      # BGR: 빨강
현재도구 = "pen"           # pen / eraser
이전점 = None
캔버스 = None

펜굵기 = 5
지우개크기 = 30

# -------------------------
# UI 그리기 함수들
# -------------------------
def 분필그리기(img, x1, y1, x2, y2, 색, 이름, 선택됨=False):
    테두리색 = (255, 255, 255) if 선택됨 else (40, 40, 40)
    두께 = 3 if 선택됨 else 1

    # 분필 몸통
    cv2.rectangle(img, (x1, y1), (x2, y2), 색, -1)

    # 분필 왼쪽 / 오른쪽 끝 약간 둥글게 느낌
    cv2.circle(img, (x1, (y1 + y2)//2), (y2-y1)//2, 색, -1)
    cv2.circle(img, (x2, (y1 + y2)//2), (y2-y1)//2, 색, -1)

    # 윤곽선
    cv2.rectangle(img, (x1, y1), (x2, y2), 테두리색, 두께)
    cv2.circle(img, (x1, (y1 + y2)//2), (y2-y1)//2, 테두리색, 두께)
    cv2.circle(img, (x2, (y1 + y2)//2), (y2-y1)//2, 테두리색, 두께)

    cv2.putText(img, 이름, (x1 + 5, y2 + 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (240, 240, 240), 1)

def 지우개그리기(img, x1, y1, x2, y2, 선택됨=False):
    테두리색 = (255, 255, 255) if 선택됨 else (30, 30, 30)
    두께 = 3 if 선택됨 else 1

    # 지우개 몸통
    cv2.rectangle(img, (x1, y1), (x2, y2), (180, 180, 210), -1)
    cv2.rectangle(img, (x1, y1), (x2, (y1+y2)//2), (120, 120, 170), -1)

    cv2.rectangle(img, (x1, y1), (x2, y2), 테두리색, 두께)

    cv2.putText(img, "ERASER", (x1 - 5, y2 + 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (240, 240, 240), 1)

def 버튼그리기(img, x1, y1, x2, y2, 글자, 선택됨=False):
    배경색 = (230, 230, 230)
    테두리색 = (255, 255, 255) if 선택됨 else (50, 50, 50)
    두께 = 3 if 선택됨 else 1

    cv2.rectangle(img, (x1, y1), (x2, y2), 배경색, -1)
    cv2.rectangle(img, (x1, y1), (x2, y2), 테두리색, 두께)
    cv2.putText(img, 글자, (x1 + 8, y1 + 33),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (20, 20, 20), 2)

def 칠판테두리그리기(img):
    h, w, _ = img.shape

    # 칠판 전체 배경
    img[:] = (45, 95, 45)   # 초록 칠판

    # 살짝 얼룩 느낌
    noise = np.random.randint(0, 10, (h, w, 3), dtype=np.uint8)
    img[:] = cv2.add(img, noise)

    # 나무 테두리
    wood = (90, 60, 30)
    cv2.rectangle(img, (0, 0), (w-1, h-1), wood, 25)

    # 안쪽 밝은 선
    cv2.rectangle(img, (18, 18), (w-19, h-19), (130, 90, 50), 2)

def 상태표시(img, 현재도구, 현재색):
    # 아래쪽 상태 표시 영역
    cv2.rectangle(img, (20, 520), (420, 570), (30, 70, 30), -1)
    cv2.rectangle(img, (20, 520), (420, 570), (220, 220, 220), 2)

    if 현재도구 == "pen":
        cv2.putText(img, "TOOL: CHALK", (35, 553),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (245, 245, 245), 2)
        cv2.rectangle(img, (270, 530), (330, 560), 현재색, -1)
        cv2.rectangle(img, (270, 530), (330, 560), (255, 255, 255), 2)
    else:
        cv2.putText(img, "TOOL: ERASER", (35, 553),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (220, 220, 220), 2)
        cv2.rectangle(img, (270, 530), (330, 560), (180, 180, 210), -1)
        cv2.rectangle(img, (270, 530), (330, 560), (255, 255, 255), 2)

# -------------------------
# 메인 루프
# -------------------------
with mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("더 남은 프레임 없다잉.")
            break

        h, w, _ = frame.shape

        if 캔버스 is None:
            # 칠판용 캔버스는 초록색으로 시작
            캔버스 = np.ones((h, w, 3), dtype=np.uint8)
            캔버스[:] = (45, 95, 45)

        # 손 인식
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        # 화면 기본판
        image = np.ones((h, w, 3), dtype=np.uint8)
        칠판테두리그리기(image)

        # 칠판 내용(사용자가 그린 것)
        image[:] = 캔버스.copy()

        # 프레임 다시 덧입히기
        wood = (90, 60, 30)
        cv2.rectangle(image, (0, 0), (w-1, h-1), wood, 25)
        cv2.rectangle(image, (18, 18), (w-19, h-19), (130, 90, 50), 2)

        # 상단 메뉴 받침 영역
        cv2.rectangle(image, (35, 30), (w-35, 110), (35, 75, 35), -1)
        cv2.rectangle(image, (35, 30), (w-35, 110), (230, 230, 230), 2)

        # 분필 / 지우개 / 클리어 버튼
        분필그리기(image, 70, 50, 120, 70, (0, 0, 255), "RED",
               선택됨=(현재도구 == "pen" and 현재색 == (0, 0, 255)))
        분필그리기(image, 180, 50, 230, 70, (0, 255, 0), "GREEN",
               선택됨=(현재도구 == "pen" and 현재색 == (0, 255, 0)))
        분필그리기(image, 300, 50, 350, 70, (255, 0, 0), "BLUE",
               선택됨=(현재도구 == "pen" and 현재색 == (255, 0, 0)))

        지우개그리기(image, 430, 45, 510, 80,
                 선택됨=(현재도구 == "eraser"))

        버튼그리기(image, 590, 42, 690, 82, "CLEAR")

        상태표시(image, 현재도구, 현재색)

        if results.multi_hand_landmarks:
            hand_landmarks = results.multi_hand_landmarks[0]
            mp_drawing.draw_landmarks(
                image, hand_landmarks, mp_hands.HAND_CONNECTIONS
            )

            검지포인트 = hand_landmarks.landmark[8]
            x = int(검지포인트.x * w)
            y = int(검지포인트.y * h)

            # 현재 도구에 따른 커서 표시
            if 현재도구 == "pen":
                cv2.circle(image, (x, y), 8, 현재색, -1)
                cv2.circle(image, (x, y), 14, (255, 255, 255), 2)
            else:
                cv2.circle(image, (x, y), 16, (180, 180, 210), -1)
                cv2.circle(image, (x, y), 16, (255, 255, 255), 2)

            # -------------------------
            # 메뉴 선택
            # -------------------------
            if 30 <= y <= 110:
                이전점 = None

                # 빨간 분필
                if 50 <= x <= 140:
                    현재색 = (0, 0, 255)
                    현재도구 = "pen"

                # 초록 분필
                elif 160 <= x <= 250:
                    현재색 = (0, 255, 0)
                    현재도구 = "pen"

                # 파란 분필
                elif 280 <= x <= 370:
                    현재색 = (255, 0, 0)
                    현재도구 = "pen"

                # 지우개
                elif 430 <= x <= 520:
                    현재도구 = "eraser"

                # 전체 삭제
                elif 590 <= x <= 690:
                    캔버스[:] = (45, 95, 45)

            # -------------------------
            # 칠판에 쓰기 / 지우기
            # -------------------------
            elif 120 < y < h - 30 and 30 < x < w - 30:
                if 이전점 is None:
                    이전점 = (x, y)
                else:
                    if 현재도구 == "pen":
                        cv2.line(캔버스, 이전점, (x, y), 현재색, 펜굵기)
                    else:
                        cv2.line(캔버스, 이전점, (x, y), (45, 95, 45), 지우개크기)

                    이전점 = (x, y)
            else:
                이전점 = None
        else:
            이전점 = None

        cv2.imshow("ELECTRONIC BLACKBOARD", image)

        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()